# 0. Configurations

In [ ]:
# Preprocessing
import os, sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Modeling
from statsmodels.discrete.discrete_model import NegativeBinomial
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy.stats import nbinom
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Robustness Checks
from scipy.stats import chi2

# Google Colab
from google.colab import drive

In [ ]:
try:
  print("✅ Running in Google Colab Environment")
  drive.mount('/content/drive', force_remount=True)
  root_dir = "/content/drive/Shareddrives/ECAIR/Innovation (Projects)/PAARAL/project_paaral/"
  PROJECT_ROOT = Path(root_dir)
  os.chdir(PROJECT_ROOT)
except:
  print("✅ Running in local Jupyter Environment")
  PROJECT_ROOT = Path.cwd().parent

if PROJECT_ROOT and str(PROJECT_ROOT) not in sys.path:
  sys.path.append(str(PROJECT_ROOT))
  print(f"✅ Added project root in system path: {PROJECT_ROOT}")

✅ Running in Google Colab Environment
Mounted at /content/drive
✅ Added project root in system path: /content/drive/Shareddrives/ECAIR/Innovation (Projects)/PAARAL/project_paaral


# 1. Load datasets

In [ ]:
data_dir = PROJECT_ROOT / 'output'
os.path.exists(data_dir)

True

In [ ]:
od_data_complete = pd.read_parquet(
    data_dir / 'od_data' / 'full_od_data.parquet'
    )

od_redirection = pd.read_parquet(data_dir / 'od_data' / 'od_redirection.parquet')

df_school_psgc = pd.read_parquet(
    os.path.join(data_dir,
                 'processed_project_bukas_school_information_psgc.parquet')
)

df_esc_rating = pd.read_parquet(data_dir / 'processed_esc_certification_rating.parquet')

df_lgu_income = pd.read_excel(
    data_dir / 'dti/2024 Raw Data.xls',
    sheet_name='Government Efficiency',
    usecols=['LOCAL GOVT UNIT', 'PROVINCE', 'REGION', 'CATEGORY',
             '14.3. Total Revenues of the LGU (in Php)']
)

df_dti_deped_matched = pd.read_excel(
    data_dir / 'dti/dti_deped_matched.xlsx',
    sheet_name='sheet_1',
)

# 2. Add data

### 2.1. Add `net_cost` and `net_cost_k`

In [ ]:
od_data_complete['net_cost'] = (od_data_complete['destination_tuition_fees'] - od_data_complete['esc_amount']).clip(lower=0.01)
od_data_complete['net_cost_k'] = od_data_complete['net_cost'] / 1_000

In [ ]:
od_redirection['net_cost'] = (od_redirection['destination_tuition_fees_algo'] - od_redirection['esc_amount']).clip(lower=0.01)
od_redirection['net_cost_k'] = od_redirection['net_cost'] / 1_000

### 2.2. Add ESC rating to `od_data_complete`

In [ ]:
esc_to_rating = dict(zip(df_esc_rating['school_id'], df_esc_rating['rating_rank']))
od_data_complete['esc_rating'] = od_data_complete['destination_school_id'].map(esc_to_rating)
od_redirection['esc_rating'] = od_redirection['private_esc_id'].map(esc_to_rating)

### 2.3. Add origin and destination region and province

In [ ]:
df_dti_deped_matched['corrected_index'] = df_dti_deped_matched['formula'] - 1
df_lgu_income['corrected_index'] = df_lgu_income.index

df_lgu_income2 = df_lgu_income.merge(df_dti_deped_matched, on='corrected_index', how='left',)[['region', 'province', 'municipality', '14.3. Total Revenues of the LGU (in Php)']]
# df_lgu_income2.head()

In [ ]:
# Manual corrections
df_lgu_income2.loc[
    df_lgu_income2['municipality'].str.contains(r'olongapo', case=False, na=False),
    'municipality'
] = 'city of olongapo'

df_lgu_income2.loc[
    df_lgu_income2['municipality'].str.contains(r'lucena city', case=False, na=False),
    'municipality'
] = 'city of lucena '

df_lgu_income2.loc[
    df_lgu_income2['municipality'].str.contains(r'kalookan', case=False, na=False),
    'municipality'
] = 'city of caloocan'

df_lgu_income2.loc[
    df_lgu_income2['municipality'].str.contains(r'taguig city', case=False, na=False),
    'municipality'
] = 'city of taguig'

In [ ]:
# Region data
school_to_region_mapping = dict(zip(df_school_psgc['school_id'], df_school_psgc['region']))

target_regions = [
    'National Capital Region (NCR)',
    'Region III (Central Luzon)',
    'Region IV-A (CALABARZON)'
]

od_redirection['destination_region'] = od_redirection['school_id_origin'].map(school_to_region_mapping)
od_redirection = od_redirection[od_redirection['destination_region'].isin(target_regions)]

# Province
school_to_province_mapping = dict(zip(df_school_psgc['school_id'], df_school_psgc['province']))
od_data_complete['origin_province'] = od_data_complete['origin_school_id'].map(school_to_province_mapping)
od_data_complete['destination_province'] = od_data_complete['destination_school_id'].map(school_to_province_mapping)

od_redirection['origin_province'] = od_redirection['school_id_origin'].map(school_to_province_mapping)
od_redirection['destination_province'] = od_redirection['school_id_destination'].map(school_to_province_mapping)

### 2.4. Add municipality income to both origin and destination

In [ ]:
school_to_lgu_mapping = dict(
    zip(df_school_psgc['school_id'],
        df_school_psgc['municipality'].str.strip().str.lower()
    )
)
lgu_to_income_mapping = dict(
    zip(
        df_lgu_income2['municipality'].str.strip().str.lower(),
        df_lgu_income2['14.3. Total Revenues of the LGU (in Php)']
    )
)

for df in [od_data_complete, od_redirection]:
    # Use the same 'school_to_lgu_mapping' for both
    # Note: Check if your columns are 'origin_school_id' or 'school_id_origin'
    orig_col = 'origin_school_id' if 'origin_school_id' in df.columns else 'school_id_origin'
    dest_col = 'destination_school_id' if 'destination_school_id' in df.columns else 'school_id_destination'

    df['origin_lgu'] = df[orig_col].map(school_to_lgu_mapping)
    df['destination_lgu'] = df[dest_col].map(school_to_lgu_mapping)

    # Map Income
    df['origin_lgu_income'] = df['origin_lgu'].map(lgu_to_income_mapping)
    df['destination_lgu_income'] = df['destination_lgu'].map(lgu_to_income_mapping)

    # Convert to numeric
    df['origin_lgu_income'] = pd.to_numeric(df['origin_lgu_income'], errors='coerce')
    df['destination_lgu_income'] = pd.to_numeric(df['destination_lgu_income'], errors='coerce')

In [ ]:
# Create a copy to preserve original data
od_data_imputed = od_data_complete.copy()

# 1. Standardize "Missingness"
# Essential: replace 0 with NaN so they don't drag the median down to zero
cols = ['origin_lgu_income', 'destination_lgu_income']
for col in cols:
    od_data_imputed[col] = od_data_imputed[col].replace(0, np.nan)

# 2. Impute by Region
# Using transform is correct/efficient.
od_data_imputed['origin_lgu_income'] = od_data_imputed['origin_lgu_income'].fillna(
    od_data_imputed.groupby('origin_province')['origin_lgu_income'].transform('median')
)
od_data_imputed['destination_lgu_income'] = od_data_imputed['destination_lgu_income'].fillna(
    od_data_imputed.groupby('destination_province')['destination_lgu_income'].transform('median')
)

print(f"Final Missing Count (Origin): {od_data_imputed['origin_lgu_income'].isna().sum()}")

Final Missing Count (Origin): 0


In [ ]:
# Create a copy to preserve original data
od_redirection_imputed = od_redirection.copy()

# 1. Standardize "Missingness"
# Essential: replace 0 with NaN so they don't drag the median down to zero
cols = ['origin_lgu_income', 'destination_lgu_income']
for col in cols:
    od_redirection_imputed[col] = od_redirection_imputed[col].replace(0, np.nan)

# 2. Impute by Region
# Using transform is correct/efficient.
od_redirection_imputed['origin_lgu_income'] = od_redirection_imputed['origin_lgu_income'].fillna(
    od_redirection_imputed.groupby('origin_province')['origin_lgu_income'].transform('median')
)
od_redirection_imputed['destination_lgu_income'] = od_redirection_imputed['destination_lgu_income'].fillna(
    od_redirection_imputed.groupby('destination_province')['destination_lgu_income'].transform('median')
)

# # 3. CRITICAL: National Fallback
# # If a region (e.g., a newly created one) has ZERO data, the step above fails.
# # We use the national median to ensure the Gravity Model doesn't crash.
# national_median = od_data_imputed['origin_lgu_income'].median()
# od_data_imputed['origin_lgu_income'] = od_data_imputed['origin_lgu_income'].fillna(national_median)
# od_data_imputed['destination_lgu_income'] = od_data_imputed['destination_lgu_income'].fillna(national_median)

print(f"Final Missing Count (Origin): {od_redirection_imputed['origin_lgu_income'].isna().sum()}")

Final Missing Count (Origin): 0


# 3. Negative Binomial Regression

In [ ]:
# Model ESC beneficiaries from public and private origin schools
od_esc_from_public = od_data_imputed[(od_data_imputed['is_beneficiary'] == 1) & (od_data_imputed['origin_sector'].isin(['Public', 'Private']))].copy()
od_esc_from_public['log_distance'] = np.log1p(od_esc_from_public['distance_km'])
od_esc_from_public['log_tuition'] = np.log1p(od_esc_from_public['tuition_fees_k'])
od_esc_from_public['net_cost_k'] = od_esc_from_public['net_cost_k'].clip(lower=0)
od_esc_from_public['log_net_cost_k'] = np.log1p(od_esc_from_public['net_cost_k'])

od_esc_from_public['log_origin_lgu_income'] = np.log1p(od_esc_from_public['origin_lgu_income'])
od_esc_from_public['log_destination_lgu_income'] = np.log1p(od_esc_from_public['destination_lgu_income'])

od_esc_from_public['esc_rating'] = od_esc_from_public['esc_rating'].fillna(0)

model_esc_public0 = NegativeBinomial.from_formula(
   'count_of_students ~ log_distance + log_net_cost_k',
    data=od_esc_from_public,
).fit(maxiter=100)

model_esc_public1 = NegativeBinomial.from_formula(
    'count_of_students ~ log_distance + log_net_cost_k + esc_rating',
    data=od_esc_from_public,
).fit(maxiter=100)

model_esc_public2 = NegativeBinomial.from_formula(
   'count_of_students ~ log_distance + log_net_cost_k + esc_rating + C(origin_region)',
    data=od_esc_from_public,
).fit(maxiter=100)

model_esc_public3 = NegativeBinomial.from_formula(
    'count_of_students ~ log_distance + log_net_cost_k + esc_rating + C(origin_region) + C(destination_region)',
    data=od_esc_from_public,
).fit(maxiter=100)

model_esc_public4 = NegativeBinomial.from_formula(
    'count_of_students ~ log_distance + log_net_cost_k + esc_rating + C(origin_region) + C(destination_region) + log_origin_lgu_income',
    data=od_esc_from_public,
).fit(maxiter=100)

model_esc_public5 = NegativeBinomial.from_formula(
    'count_of_students ~ log_distance + log_net_cost_k + esc_rating + C(origin_region) + C(destination_region) + log_origin_lgu_income',
    data=od_esc_from_public,
).fit(maxiter=100)

model_esc_public6 = NegativeBinomial.from_formula(
     'count_of_students ~ log_distance + log_net_cost_k + esc_rating + C(origin_region) + C(destination_region) + log_origin_lgu_income + log_destination_lgu_income',
    data=od_esc_from_public,
).fit(maxiter=100)

Optimization terminated successfully.
         Current function value: 1.842269
         Iterations: 14
         Function evaluations: 15
         Gradient evaluations: 15
Optimization terminated successfully.
         Current function value: 1.842226
         Iterations: 14
         Function evaluations: 16
         Gradient evaluations: 16
Optimization terminated successfully.
         Current function value: 1.840958
         Iterations: 18
         Function evaluations: 20
         Gradient evaluations: 20
Optimization terminated successfully.
         Current function value: 1.840611
         Iterations: 23
         Function evaluations: 24
         Gradient evaluations: 24
Optimization terminated successfully.
         Current function value: 1.835938
         Iterations: 33
         Function evaluations: 36
         Gradient evaluations: 36
Optimization terminated successfully.
         Current function value: 1.835938
         Iterations: 33
         Function evaluations: 36
  

# 4. Robustness Checks

In [ ]:
def evaluate_model_hierarchy(models, names):
    results = []

    for i in range(len(models)):
        current_model = models[i]
        name = names[i]

        # Calculate LRT against the immediate previous model (nested)
        if i > 0:
            previous_model = models[i-1]
            llr_stat = 2 * (current_model.llf - previous_model.llf)
            df_diff = current_model.df_model - previous_model.df_model
            # Handle potential numerical issues where llr might be slightly negative
            p_val = chi2.sf(max(0, llr_stat), df_diff)
        else:
            llr_stat, p_val = None, None

        results.append({
            'Model Name': name,
            'Log-Likelihood': current_model.llf,
            'AIC': current_model.aic,
            'BIC': current_model.bic,
            'LRT Stat': llr_stat,
            'LRT p-value': p_val,
            'Alpha (Dispersion)': current_model.params['alpha']
        })

    return pd.DataFrame(results)

# Define your model suite and labels
models = [model_esc_public0, model_esc_public1, model_esc_public2,
          model_esc_public3, model_esc_public4, model_esc_public5, model_esc_public6]

names = ['Baseline', '+ Rating', '+ Origin Region', '+ Dest Region',
         '+ Origin Income Only', '+ Dest Income Only', '+ Dual Income']

comparison_df = evaluate_model_hierarchy(models, names)

In [ ]:
comparison_df

,Model Name,Log-Likelihood,AIC,BIC,LRT Stat,LRT p-value,Alpha (Dispersion)
0,Baseline,-53838.457513,107684.915025,107718.046007,NaN,NaN,0.402239
1,+ Rating,-53837.212371,107684.424743,107725.838470,2.490283,1.145512e-01,0.402113
2,+ Origin Region,-53800.148922,107614.297845,107672.277064,74.126898,8.008454e-17,0.401387
3,+ Dest Region,-53790.010011,107598.020023,107672.564733,20.277822,3.951181e-05,0.400717
4,+ Origin Income Only,-53653.453412,107326.906824,107409.734279,273.113199,2.379114e-61,0.394777
5,+ Dest Income Only,-53653.453412,107326.906824,107409.734279,0.000000,NaN,0.394777
6,+ Dual Income,-53592.411323,107206.822647,107297.932848,122.084177,2.212488e-28,0.392479


In [ ]:
# 1. Pre-process: Group by cluster once to avoid filtering inside the loop
cluster_dict = {cid: group for cid, group in od_esc_from_public.groupby('origin_school_id')}
cluster_ids = list(cluster_dict.keys())

n_iterations = 1_000
boot_results = []

print(f"Starting Optimized Cluster Bootstrap (n={n_iterations}) for 7,392 clusters...")

for i in range(n_iterations):
    # 2. Resample Cluster IDs with replacement
    resampled_ids = np.random.choice(cluster_ids, size=len(cluster_ids), replace=True)

    # 3. Efficiently concatenate the groups
    # This is still the heavy part, but pre-grouping makes it faster
    boot_df = pd.concat([cluster_dict[cid] for cid in resampled_ids], ignore_index=True)

    try:
        # 4. Re-fit Model 6
        # Fixing alpha at 0.3925 ensures we focus on the stability of the slopes
        model = smf.glm(formula='count_of_students ~ log_distance + log_net_cost_k + esc_rating + C(origin_region) + C(destination_region) + log_origin_lgu_income + log_destination_lgu_income',
                        data=boot_df,
                        family=sm.families.NegativeBinomial(alpha=0.3925)).fit()

        boot_results.append(model.params)

        if (i + 1) % 50 == 0:
            print(f"Iteration {i + 1} complete...")

    except Exception as e:
        continue

# 5. Analyze Results for All Coefficients
boot_results_df = pd.DataFrame(boot_results)
ci_summary = boot_results_df.quantile([0.025, 0.5, 0.975]).T
ci_summary.columns = ['Lower 2.5%', 'Median', 'Upper 97.5%']

Starting Optimized Cluster Bootstrap (n=1000) for 7,392 clusters...
Iteration 50 complete...
Iteration 100 complete...
Iteration 150 complete...
Iteration 200 complete...
Iteration 250 complete...
Iteration 300 complete...
Iteration 350 complete...
Iteration 400 complete...
Iteration 450 complete...
Iteration 500 complete...
Iteration 550 complete...
Iteration 600 complete...
Iteration 650 complete...
Iteration 700 complete...
Iteration 750 complete...
Iteration 800 complete...
Iteration 850 complete...
Iteration 900 complete...


KeyboardInterrupt: 

In [ ]:
print("\n95% Bootstrap Confidence Intervals:")
print(ci_summary)

# 5. Generating the candidate pool

In [ ]:
# Define the untapped (latent) demand from the true OD
# We use 'latent' to describe studens who aren't yet beneficiaries
od_potential = od_esc_from_public.copy()
od_potential['candidate_pool'] = (od_potential['origin_g6_enrollment'] -
                               od_potential['count_of_students'])
od_potential = od_potential[[
    'origin_school_id', 'destination_school_id', 'origin_sector',
    'origin_region', 'destination_sector', 'tuition_fees_k',
    'destination_tuition_fees', 'distance_meters', 'distance_km',
    'net_cost', 'net_cost_k', 'esc_rating', 'log_distance',
    'log_tuition', 'log_net_cost_k', 'destination_jhs_enrollment',
    'esc_amount', 'esc_amount_k', 'candidate_pool', 'log_origin_lgu_income',
    'log_destination_lgu_income', 'scaled_origin_income',
    'scaled_dest_income', 'destination_region',
]]

# Keep only paths where potential movers actually exist
od_potential = od_potential[od_potential['candidate_pool'] > 0].reset_index(drop=True)

######## Process untapped demand from new OD matrix
od_redirection_imputed['log_distance'] = np.log1p(od_redirection_imputed['distance_to_private_km'])
od_redirection_imputed['log_tuition'] = np.log1p(od_redirection_imputed['destination_tuition_fees_algo_k'])
od_redirection_imputed['log_net_cost_k'] = np.log1p(od_redirection_imputed['net_cost_k'])

od_redirection_imputed['log_origin_lgu_income'] = np.log1p(od_redirection_imputed['origin_lgu_income'])
od_redirection_imputed['log_destination_lgu_income'] = np.log1p(od_redirection_imputed['destination_lgu_income'])

od_redirection_imputed['esc_rating'] = od_redirection_imputed['esc_rating'].fillna(0)

od_redirection_potential = od_redirection_imputed[
    ['school_id_origin', 'private_esc_id', 'distance_to_private_m',
     'distance_to_private_km', 'origin_region', 'origin_sector',
     'destination_sector_algo', 'count_non_beneficiary', 'enrollment_jhs',
     'esc_amount', 'esc_amount_k', 'net_cost', 'net_cost_k', 'esc_rating',
     'destination_tuition_fees_algo', 'destination_tuition_fees_algo_k',
     'log_distance', 'log_tuition', 'log_net_cost_k',
     'log_origin_lgu_income', 'log_destination_lgu_income',
     'scaled_origin_income', 'scaled_dest_income', 'destination_region',]
]

od_redirection_potential = od_redirection_potential.rename(columns={
    'school_id_origin': 'origin_school_id',
    'private_esc_id': 'destination_school_id',
    'distance_to_private_m': 'distance_meters',
    'distance_to_private_km': 'distance_km',
    'destination_sector_algo': 'destination_sector',
    'count_non_beneficiary': 'candidate_pool',
    'enrollment_jhs': 'destination_jhs_enrollment',
    'destination_tuition_fees_algo': 'destination_tuition_fees',
    'destination_tuition_fees_algo_k': 'tuition_fees_k',
})

# Sanity check: The two OD matrices should have the same columns
if not set(od_potential.columns).symmetric_difference(set(od_redirection_potential.columns)):
  candidate_beneficiary_pool = pd.concat([od_potential,
                                          od_redirection_potential],
                                         axis=0, sort=False, ignore_index=True)

# 7. Generating scenarios

In [ ]:
# Scenario: Baseline (Status Quo - Zero Subsidy)
baseline_features = candidate_beneficiary_pool.copy()
baseline_features['esc_amount_k'] = 0
baseline_features['net_cost_k'] = (baseline_features['tuition_fees_k'] - baseline_features['esc_amount_k']).clip(lower=0.01)
baseline_features['log_net_cost_k'] = np.log1p(baseline_features['net_cost_k'])

# Predicted baseline mu (expected flow if no help is given)
candidate_beneficiary_pool['mu_baseline_0_subsidy'] = model_esc_public4.predict(baseline_features)

########

# Scenario: Policy Intervention
# Non-beneficiaries will get the same subsidy amount currently given to beneficiaries on a specific OD path
policy_features = baseline_features.copy()
policy_features['esc_amount_k'] = candidate_beneficiary_pool['esc_amount_k']
policy_features['net_cost_k'] = (policy_features['tuition_fees_k'] - policy_features['esc_amount_k']).clip(lower=0.01)
policy_features['log_net_cost_k'] = np.log1p(policy_features['net_cost_k'])

# Predicted policy mu (expected flow with as-is subsidy)
candidate_beneficiary_pool['mu_with_current_subsidy'] = model_esc_public4.predict(policy_features)

########

# Scenario: Decrease in net_cost (Increase in subsidy) by 1,000 pesos
minus_1k_features = baseline_features.copy()
minus_1k_features['esc_amount_k'] = candidate_beneficiary_pool['esc_amount_k'] + 1
minus_1k_features['net_cost_k'] = (minus_1k_features['tuition_fees_k'] - minus_1k_features['esc_amount_k']).clip(lower=0.01)
minus_1k_features['log_net_cost_k'] = np.log1p(minus_1k_features['net_cost_k'])

# Predicted policy mu (expected flow with a 1,000-peso decrease in net cost/1,000-peso increase in subsidy)
candidate_beneficiary_pool['mu_with_minus_1k_net_cost'] = model_esc_public4.predict(minus_1k_features)

########

# Scenario: Decrease in net_cost (Increase in subsidy) by 10,000 pesos
minus_10k_features = baseline_features.copy()
minus_10k_features['esc_amount_k'] = candidate_beneficiary_pool['esc_amount_k'] + 10
minus_10k_features['net_cost_k'] = (minus_10k_features['tuition_fees_k'] - minus_10k_features['esc_amount_k']).clip(lower=0.01)
minus_10k_features['log_net_cost_k'] = np.log1p(minus_10k_features['net_cost_k'])

# Predicted policy mu (expected flow with a 10,000-peso decrease in net cost/10,000-peso increase in subsidy)
candidate_beneficiary_pool['mu_with_minus_10k_net_cost'] = model_esc_public4.predict(minus_10k_features)

########

# Scenario: Decrease in net_cost (Increase in subsidy) by 15,000 pesos
minus_15k_features = baseline_features.copy()
minus_15k_features['esc_amount_k'] = candidate_beneficiary_pool['esc_amount_k'] + 15
minus_15k_features['net_cost_k'] = (minus_15k_features['tuition_fees_k'] - minus_15k_features['esc_amount_k']).clip(lower=0.01)
minus_15k_features['log_net_cost_k'] = np.log1p(minus_15k_features['net_cost_k'])

# Predicted policy mu (expected flow with a 15,000-peso decrease in net cost/15,000-peso increase in subsidy)
candidate_beneficiary_pool['mu_with_minus_15k_net_cost'] = model_esc_public4.predict(minus_15k_features)

########

# Scenario: Decrease in net_cost (Increase in subsidy) by 20,000 pesos
minus_20k_features = baseline_features.copy()
minus_20k_features['esc_amount_k'] = candidate_beneficiary_pool['esc_amount_k'] + 20
minus_20k_features['net_cost_k'] = (minus_20k_features['tuition_fees_k'] - minus_20k_features['esc_amount_k']).clip(lower=0.01)
minus_20k_features['log_net_cost_k'] = np.log1p(minus_20k_features['net_cost_k'])

# Predicted policy mu (expected flow with a 20,000-peso decrease in net cost/20,000-peso increase in subsidy)
candidate_beneficiary_pool['mu_with_minus_20k_net_cost'] = model_esc_public4.predict(minus_20k_features)

########

# Mapping the net cost back to the main dataframe for comparison
candidate_beneficiary_pool['net_cost_baseline'] = baseline_features['net_cost_k']
candidate_beneficiary_pool['net_cost_current']  = policy_features['net_cost_k']
candidate_beneficiary_pool['net_cost_minus_1k'] = minus_1k_features['net_cost_k']
candidate_beneficiary_pool['net_cost_minus_10k']= minus_10k_features['net_cost_k']
candidate_beneficiary_pool['net_cost_minus_15k']= minus_15k_features['net_cost_k']
candidate_beneficiary_pool['net_cost_minus_20k']= minus_20k_features['net_cost_k']

In [ ]:
def simulate_policy_scenario(df, model, subsidy_increment_k, suffix):
    """
    Generates net cost and predicted student flow (mu) for a given policy scenario.

    Parameters:
    - df: DataFrame containing the candidate pool and baseline features.
    - model: The trained NegativeBinomial model (e.g., model_esc_public4).
    - subsidy_increment_k: The amount (in thousands) to add to the existing subsidy.
    - suffix: The string suffix for the new column names (e.g., 'minus_1k').

    Returns:
    - A DataFrame with the new 'net_cost_' and 'mu_' columns added.
    """
    # Create a temporary feature set for prediction
    temp_features = df.copy()

    # Update subsidy and calculate new net cost (ensuring it doesn't go below 0.01)
    # Note: If baseline (0 subsidy) is needed, pass subsidy_increment_k as a
    # value that offsets the existing 'esc_amount_k' to zero.
    new_esc_amount = (df['esc_amount_k'] + subsidy_increment_k).clip(lower=0)
    temp_features['net_cost_k'] = (temp_features['tuition_fees_k'] - new_esc_amount).clip(lower=0)

    # Transform friction variable
    temp_features['log_net_cost_k'] = np.log1p(temp_features['net_cost_k'])

    # Predict expected flow (mu)
    df[f'net_cost_{suffix}'] = temp_features['net_cost_k']
    df[f'mu_{suffix}'] = model.predict(temp_features)

    return df

# 1. Baseline: Zero Subsidy (Assuming current esc_amount_k exists in df)
# We subtract the current amount to reach zero
candidate_beneficiary_pool = simulate_policy_scenario(
    candidate_beneficiary_pool,
    # model_esc_public4,
    results_clustered,
    subsidy_increment_k=-candidate_beneficiary_pool['esc_amount_k'],
    suffix='baseline_0_subsidy'
)

# 2. Current Subsidy (Increment is 0)
candidate_beneficiary_pool = simulate_policy_scenario(
    candidate_beneficiary_pool,
    # model_esc_public4,
    results_clustered,
    0,
    'current_subsidy'
)

for amt in list(range(1,21)):
  candidate_beneficiary_pool = simulate_policy_scenario(
      candidate_beneficiary_pool,
      # model_esc_public4,
      results_clustered,
      amt,
      f'minus_{amt}k_net_cost'
  )

In [ ]:
candidate_beneficiary_pool.to_parquet(data_dir / 'full_candidate_beneficiary_pool_without_probdist_0209_clusteredSE.parquet.parquet', index=False)

# X. Scratch

### Distance-binned elasticities

In [ ]:
od_esc_from_public['distance_km']

,distance_km
0,215.1594
1,216.8214
2,95.0352
3,74.2692
4,254.1570
...,...
29219,0.8877
29220,2.8531
29221,16.9525
29222,1.0270


In [ ]:
df = od_esc_from_public.copy()

In [ ]:
# df['distance_bin'] = pd.qcut(df['distance_km'], q=5, labels=['Q1', 'Q2', 'Q3',
#                                                              'Q4', 'Q5'])

df['distance_bin'] = pd.cut(df['distance_km'],
                            bins=[0, 3, 7, 15, 30, np.inf],
                            labels=['0-3km','3-7km','7-15km','15-30km', '30km+'])

In [ ]:
#Create bin dummies
bin_dummies = pd.get_dummies(df['distance_bin'], prefix='bin', drop_first=False)
df = pd.concat([df, bin_dummies], axis=1)

# Create interaction terms
bin_cols = [c for c in df.columns if c.startswith('bin_')]
for col in bin_cols:
  df[f'{col}_x_lncost'] = df[col] * df['log_net_cost_k']

df = df.rename(columns={
    'bin_0-3km_x_lncost':   'bin_0_3km_x_lncost',
    'bin_3-7km_x_lncost':   'bin_3_7km_x_lncost',
    'bin_7-15km_x_lncost':  'bin_7_15km_x_lncost',
    'bin_15-30km_x_lncost': 'bin_15_30km_x_lncost',
    'bin_30km+_x_lncost':   'bin_30kmp_x_lncost'
})

In [ ]:
df

,origin_school_id,destination_school_id,count_of_students,destination_region,is_beneficiary,origin_region,origin_sector,destination_sector,destination_tuition_fees,distance_meters,...,bin_0-3km,bin_3-7km,bin_7-15km,bin_15-30km,bin_30km+,bin_0_3km_x_lncost,bin_3_7km_x_lncost,bin_7_15km_x_lncost,bin_15_30km_x_lncost,bin_30kmp_x_lncost
0,104408,401116,1,Region III (Central Luzon),1,Region III (Central Luzon),Public,Private,18525.00,215159.4,...,False,False,False,False,True,0.000000,0.0,0.0,0.00000,2.353753
1,104412,401116,1,Region III (Central Luzon),1,Region III (Central Luzon),Public,Private,18525.00,216821.4,...,False,False,False,False,True,0.000000,0.0,0.0,0.00000,2.353753
2,104457,401070,1,Region III (Central Luzon),1,Region III (Central Luzon),Public,Private,13350.00,95035.2,...,False,False,False,False,True,0.000000,0.0,0.0,0.00000,1.677097
3,104459,401365,1,Region III (Central Luzon),1,Region III (Central Luzon),Public,Private,25475.04,74269.2,...,False,False,False,False,True,0.000000,0.0,0.0,0.00000,2.860774
4,104460,402529,1,Region IV-A (CALABARZON),1,Region III (Central Luzon),Public,Private,21875.00,254157.0,...,False,False,False,False,True,0.000000,0.0,0.0,0.00000,2.630089
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29219,500567,406291,1,National Capital Region (NCR),1,National Capital Region (NCR),Public,Private,43900.00,887.7,...,True,False,False,False,False,3.462606,0.0,0.0,0.00000,0.000000
29220,500567,406295,1,National Capital Region (NCR),1,National Capital Region (NCR),Public,Private,44576.00,2853.1,...,True,False,False,False,False,3.483576,0.0,0.0,0.00000,0.000000
29221,500567,406678,1,National Capital Region (NCR),1,National Capital Region (NCR),Public,Private,52640.30,16952.5,...,False,False,False,True,False,0.000000,0.0,0.0,3.70476,0.000000
29222,500781,403214,1,Region IV-A (CALABARZON),1,Region IV-A (CALABARZON),Public,Private,35160.55,1027.0,...,True,False,False,False,False,3.301766,0.0,0.0,0.00000,0.000000


In [ ]:
df.columns
# Check your current interaction column names
print([c for c in df.columns if 'x_lncost' in c])

['bin_0-3km_x_lncost', 'bin_3-7km_x_lncost', 'bin_7-15km_x_lncost', 'bin_15-30km_x_lncost', 'bin_30km+_x_lncost']


In [ ]:
# Re-estimate gravity model
# interaction_terms = ' + '.join([
#     f'bin_{b}_x_lncost'
#     for b in ['Q2', 'Q3', 'Q4', 'Q5']
# ])
interaction_terms = ' + '.join([
    'bin_3_7km_x_lncost',
    'bin_7_15km_x_lncost',
    'bin_15_30km_x_lncost',
    'bin_30kmp_x_lncost'
])

formula = f"""
  count_of_students ~ log_distance
                    + log_net_cost_k
                    + {interaction_terms}
                    + log_origin_lgu_income
                    + log_destination_lgu_income
                    + esc_rating
                    + C(origin_region)
                    + C(destination_region)
"""

# Fit negative binomial
model = smf.negativebinomial(formula, data=df).fit(
    # method='newton',
    cov_type='cluster',
    cov_kwds={'groups': df['origin_school_id']},
    maxiter=200,
)

print(model.summary())

Optimization terminated successfully.
         Current function value: 1.809041
         Iterations: 37
         Function evaluations: 41
         Gradient evaluations: 41
                     NegativeBinomial Regression Results                      
Dep. Variable:      count_of_students   No. Observations:                29224
Model:               NegativeBinomial   Df Residuals:                    29210
Method:                           MLE   Df Model:                           13
Date:                Fri, 13 Mar 2026   Pseudo R-squ.:                 0.09861
Time:                        07:05:43   Log-Likelihood:                -52867.
converged:                       True   LL-Null:                       -58651.
Covariance Type:              cluster   LLR p-value:                     0.000
                                                          coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------

In [ ]:
-0.1297 + (-0.0443)
# -0.1297 + 0.0131
-0.1297 + 0.3087

0.17899999999999996

In [ ]:
# Standardized coefficient = elasticity × (std of predictor / std of outcome)
# For log-log models, a simpler approach is to compare
# the effect of a 1 standard deviation change in each variable

log_distance_std = df['log_distance'].std()
log_cost_std = df['log_net_cost_k'].std()

# Effect of 1 SD change in distance (same for all bins)
effect_distance = -0.6229 * log_distance_std

# Effect of 1 SD change in cost, per bin
bin_elasticities = {
    '0-3km':   -0.1297,
    '3-7km':   -0.1740,
    '7-15km':  -0.1166,
    '15-30km': -0.0252,
    '30km+':   +0.1790
}

print(f"Distance effect (1 SD): {effect_distance:.4f}")
print()
for bin_name, elasticity in bin_elasticities.items():
    effect_cost = elasticity * log_cost_std
    ratio = abs(effect_distance / effect_cost) if effect_cost != 0 else float('inf')
    print(f"{bin_name}: cost effect = {effect_cost:.4f}, distance/cost ratio = {ratio:.1f}x")

Distance effect (1 SD): -0.8080

0-3km: cost effect = -0.1045, distance/cost ratio = 7.7x
3-7km: cost effect = -0.1402, distance/cost ratio = 5.8x
7-15km: cost effect = -0.0940, distance/cost ratio = 8.6x
15-30km: cost effect = -0.0203, distance/cost ratio = 39.8x
30km+: cost effect = 0.1443, distance/cost ratio = 5.6x


In [ ]:
log_cost_std

0.8059514602827275